# Stage 1 / Step 3+4 - XGBoost fusion

Combine the three component signals into one viral prediction:
- Component 1: `content_score` (TF-IDF text model, out-of-fold)
- Component 2: `topic_score` (BERTopic trend weight, label-free)
- Component 3: author / context / structure features (channel, timing, duration, text structure)

Historical target: `is_viral` used the retired relative top-25% rule. This notebook is not an official dataset-v3 run. Evaluated out-of-fold with PR-AUC (matches the team's
diagram metric). SHAP shows which signals drive the final decision.

In [ ]:
# assemble the fusion feature matrix from the 3 components
from pathlib import Path
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report
import shap
import joblib

ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
feat = pd.read_parquet(ROOT / "ml" / "data" / "video_features.parquet")
c1 = pd.read_parquet(ROOT / "ml" / "data" / "stage1_scores.parquet")      # content_score, is_viral
c2 = pd.read_parquet(ROOT / "ml" / "data" / "stage2_topic_scores.parquet") # topic_score

df = feat.merge(c1, on="video_id").merge(c2[["video_id", "topic_score"]], on="video_id")

author_context = [
    "channel_freq", "duration_min", "pub_hour", "pub_dow", "pub_month",
    "title_len_words", "desc_len_words", "trans_len_words",
    "title_sentiment", "title_has_question", "title_upper_ratio",
    "kw_price", "kw_range", "kw_charging", "has_description", "has_transcript",
]
fusion_features = ["content_score", "topic_score"] + author_context
X = df[fusion_features]
y = df["is_viral"]
print("fusion features:", len(fusion_features), "| viral rate:", round(y.mean(), 3))

In [ ]:
# XGBoost fusion, evaluated out-of-fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
xgb = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=3,
                    eval_metric="logloss", random_state=42, n_jobs=-1)

oof = cross_val_predict(xgb, X, y, cv=cv, method="predict_proba")[:, 1]
ap, auc = average_precision_score(y, oof), roc_auc_score(y, oof)
print(f"FUSION (3 components)   PR-AUC: {ap:.3f}  |  ROC-AUC: {auc:.3f}")
print(f"content_score alone     PR-AUC: 0.604  (reference)")
print(f"baseline (viral rate)   PR-AUC: {y.mean():.3f}")
print("\n", classification_report(y, (oof >= 0.5).astype(int), target_names=["not-viral", "viral"]))

In [ ]:
# SHAP: which signals drive the fused prediction
xgb.fit(X, y)
explainer = shap.TreeExplainer(xgb)
sv = explainer.shap_values(X)
imp = (pd.DataFrame({"feature": fusion_features, "mean_abs_shap": np.abs(sv).mean(axis=0)})
       .sort_values("mean_abs_shap", ascending=False))
print("=== Drivers of the fused viral prediction (SHAP) ===")
print(imp.to_string(index=False))

In [ ]:
# save final fused model + OOF probabilities
joblib.dump({"model": xgb, "features": fusion_features}, ROOT / "ml" / "models" / "stage1_fusion.joblib")
out = df[["video_id", "is_viral", "content_score", "topic_score"]].copy()
out["fusion_viral_proba"] = oof
out.to_parquet(ROOT / "ml" / "data" / "stage1_final_scores.parquet", index=False)
print("Saved stage1_fusion.joblib + stage1_final_scores.parquet")